# Peltonova lopatica: predvidi → izračunaj → provjeri

Mlaz brzine \(c_1\) sustiže lopaticu obodne brzine \(u\). Relativna ulazna brzina je \(w_1=c_1-u\). Simetrična Peltonova zdjelica poništava poprečne komponente dviju polovica mlaza; računamo tangencijalnu silu i snagu.

## Predvidi

1. Je li snaga najveća kada lopatica miruje, kada se giba kao mlaz ili između tih granica?
2. Kako gubitak relativne brzine utječe na optimalni omjer \(u/c_1\), a kako na najveću snagu?
3. Koji granični slučajevi moraju dati nultu snagu?


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

def pelton(c1, u, Q, rho=998.0, k=0.90, outlet_deviation_deg=15.0):
    # k je omjer iznosa izlazne i ulazne relativne brzine.
    c1, u = np.broadcast_arrays(c1, u)
    if np.any(c1 <= 0) or np.any((u < 0) | (u > c1)) or not (0 <= k <= 1):
        raise ValueError("Treba vrijediti c1>0, 0≤u≤c1 i 0≤k≤1.")
    phi = np.deg2rad(outlet_deviation_deg)
    w1 = c1-u
    c2_t = u-k*w1*np.cos(phi)
    force_t = rho*Q*(c1-c2_t)
    power = force_t*u
    jet_power = 0.5*rho*Q*c1**2
    return force_t, power, power/jet_power

c1, Q, k, phi = 36.0, 0.045, 0.90, 15.0
u_grid = np.linspace(0, c1, 1001)
force, power, efficiency = pelton(c1, u_grid, Q, k=k, outlet_deviation_deg=phi)
i_opt = int(np.argmax(power))
u_opt_num = u_grid[i_opt]
u_opt_analytic = c1/2
print(f"Numerički optimum u = {u_opt_num:.3f} m/s = {u_opt_num/c1:.3f} c1")
print(f"P_max = {power[i_opt]/1000:.3f} kW; eta_mlaz→rotor = {efficiency[i_opt]:.3f}")


## Izračunaj: optimiranje i osjetljivost

Snagu tražimo pretragom mreže, bez unošenja poznatog optimuma u algoritam. Nakon toga mijenjamo koeficijent očuvanja relativne brzine \(k\) i odstupanje izlaza od idealnog okreta. To razdvaja **položaj optimuma** od **vrijednosti optimuma**.


In [ ]:
k_values = [0.80, 0.90, 1.00]
phi_values = np.linspace(0, 30, 61)
eta_max = np.empty((len(k_values), len(phi_values)))
for i, k_i in enumerate(k_values):
    for j, phi_i in enumerate(phi_values):
        eta_max[i,j] = pelton(c1, c1/2, Q, k=k_i, outlet_deviation_deg=phi_i)[2]

# Neovisna lokalna provjera stacionarnosti: centrirana derivacija snage.
du = 1e-3*c1
p_plus = pelton(c1, c1/2+du, Q, k=k, outlet_deviation_deg=phi)[1]
p_minus = pelton(c1, c1/2-du, Q, k=k, outlet_deviation_deg=phi)[1]
dP_du = (p_plus-p_minus)/(2*du)
print(f"Numerička derivacija dP/du u c1/2: {float(dP_du):.3e} N")
print(f"Pad eta_max zbog k: 1.00→0.80 pri φ=15°: {eta_max[-1,30]-eta_max[0,30]:.3f}")


## Provjeri

Provjere koriste optimalnost, granične slučajeve i energetsku granicu. U ovom pojednostavljenom modelu nema mehaničkih gubitaka rotora; zato dobivena učinkovitost nije ukupna učinkovitost turbine.


In [ ]:
assert np.isclose(u_opt_num, u_opt_analytic, atol=c1/1000)
assert abs(float(dP_du)) < 1e-7*power[i_opt]/c1
assert np.isclose(pelton(c1, 0.0, Q, k=k, outlet_deviation_deg=phi)[1], 0.0)
assert np.isclose(pelton(c1, c1, Q, k=k, outlet_deviation_deg=phi)[1], 0.0)
assert np.all((eta_max >= 0) & (eta_max <= 1+1e-12))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for k_i in k_values:
    p_i = pelton(c1, u_grid, Q, k=k_i, outlet_deviation_deg=phi)[1]
    axes[0].plot(u_grid/c1, p_i/1000, label=f"k={k_i:.2f}")
axes[0].axvline(.5, color="#b43c35", ls="--", label="u/c1=0,5")
axes[0].set(xlabel="$u/c_1$", ylabel="snaga (kW)", title="Numeričko traženje optimuma")
axes[0].legend()
for i, k_i in enumerate(k_values):
    axes[1].plot(phi_values, eta_max[i], label=f"k={k_i:.2f}")
axes[1].set(xlabel="odstupanje izlaza φ (°)", ylabel="najveća učinkovitost mlaza", title="Osjetljivost na gubitak i kut")
axes[1].legend()
for ax in axes: ax.grid(True, ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Protumači

Dodaj ograničenje najveće dopuštene obodne brzine. Ako je manje od \(c_1/2\), matematički optimum više nije ostvariv i inženjerski optimum leži na granici dopuštenog područja.
